# Data Preparation Lab -- Module 2, Class 2

**Dataset:** Superstore Sales

In this lab you will:
1. Load and inspect data
2. Handle missing values
3. Remove duplicates
4. Convert data types
5. Create derived features

The first 3 tasks are pre-built. The rest are TODO for you.

---
## Setup: Load the Dataset

Option A: Upload from Kaggle (download from https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

Option B: Use the URL loader below.

In [ ]:
import pandas as pd
import numpy as np

# Option A: Upload in Colab
# from google.colab import files
# uploaded = files.upload()  # upload your CSV
# df = pd.read_csv('SampleSuperstore.csv')

# Option B: Load from a public URL
# If the URL does not work, use Option A.
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv"

try:
    df = pd.read_csv(url, encoding='latin-1')
    print(f"Loaded from URL: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print(f"URL failed ({e}). Use Option A: upload the CSV manually.")
    # Fallback: upload manually
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Loaded from upload: {df.shape[0]} rows, {df.shape[1]} columns")

---
## Task 1: Inspect the Data (pre-built)

Always look at your data before doing anything to it.

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Shape: rows x columns
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe()

---
## Task 2: Missing Values (pre-built)

Check which columns have missing values and how many.

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

# Show only columns with missing values
missing_report[missing_report['missing_count'] > 0]

In [ ]:
# Fill missing numerical values with median (robust to outliers)
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} missing values with median: {median_val}")

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} missing values with mode: {mode_val}")

# Verify: no more missing values
print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")

---
## Task 3: Remove Duplicates (pre-built)

In [ ]:
# Check for duplicates
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {n_duplicates}")

# Remove duplicates
if n_duplicates > 0:
    df = df.drop_duplicates()
    print(f"After removal: {df.shape[0]} rows remain")
else:
    print("No duplicates to remove.")

---
## Task 4: Convert Date Columns (TODO)

The `Order Date` and `Ship Date` columns are stored as strings. Convert them to proper datetime objects.

Hint: Use `pd.to_datetime()`. After conversion, verify with `.dtypes`.

In [ ]:
# Check current types of date columns
print("Before conversion:")
for col in df.columns:
    if 'date' in col.lower() or 'Date' in col:
        print(f"  {col}: {df[col].dtype}")
        print(f"  Sample value: {df[col].iloc[0]}")

In [ ]:
# pd.to_datetime() is a Pandas function that converts a column of text (strings)
# into proper datetime objects that Python understands as actual dates.
# Without this conversion, dates are just plain text — you can't do date math on them.
#
# df['Order Date'] refers to the 'Order Date' column inside the DataFrame df.
# The = sign reassigns that column, replacing the old text values with datetime values.

df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print("Date columns converted successfully!")
# print() displays a message in the output so we can confirm the code ran.


In [ ]:
# After converting, we verify the result in two ways:

# --- Verification 1: check the data types ---
# df.columns is a list of all column names in the DataFrame.
# The for loop iterates (goes one-by-one) through every column name.
# 'col' is a temporary variable that holds the current column name on each loop.
# .lower() converts the column name to lowercase so the check works regardless of capitalisation.
# 'in' checks whether a substring exists inside the string.
# df[col].dtype tells us the data type of that column (e.g., object, datetime64, int64).
print("After conversion — data types:")
for col in df.columns:
    if 'date' in col.lower():
        print(f"  {col}: {df[col].dtype}")
        # f-string: the f before the quote lets us embed variables inside {} directly in the string.

# --- Verification 2: spot-check actual values ---
# df[['Order Date', 'Ship Date']] selects only those two columns (double brackets = multiple columns).
# .head() returns the first 5 rows by default so we can visually confirm dates look correct.
df[['Order Date', 'Ship Date']].head()


---
## Task 5: Derived Features (TODO)

Create customer-level summary features. These are the building blocks for customer segmentation (Activity 4).

You need to create:
- **total_spending**: Total sales per customer
- **order_frequency**: Number of orders per customer
- **avg_order_value**: Average sales amount per order per customer

Hint: Use `df.groupby('Customer ID')` (or whatever the customer ID column is named).

In [ ]:
# First, identify the right column names
print("All columns:")
for col in df.columns:
    print(f"  {col}")

In [ ]:
# --- Total Spending per Customer ---

# df.groupby('Customer ID') splits the DataFrame into groups,
# one group per unique Customer ID. Think of it like sorting receipts by customer.
#
# ['Sales'] selects only the Sales column within each group.
#
# .sum() adds up all the Sales values for each customer group,
# giving a single total number per customer.
#
# The result is a Series (a single column of data) where:
#   - the index (row labels) = Customer IDs
#   - the values = total sales for that customer

customer_spending = df.groupby('Customer ID')['Sales'].sum()

# .name sets the label of the Series — this becomes the column name when we merge later.
customer_spending.name = 'total_spending'

# .sort_values(ascending=False) sorts from largest to smallest (descending order).
# .head() shows only the top 5 results.
print("Top 5 customers by total spending:")
print(customer_spending.sort_values(ascending=False).head())


In [ ]:
# --- Order Frequency per Customer ---

# Again we groupby 'Customer ID' to work per customer.
#
# ['Order ID'] selects the Order ID column within each group.
#
# .nunique() stands for 'number of unique values'.
# We use nunique() instead of .count() because one order can span
# multiple rows (one row per product). nunique() counts each Order ID
# only once, giving us the true number of separate orders.
#
# Example: if a customer has 10 rows but only 3 unique Order IDs,
# they placed 3 orders — not 10.
order_freq = df.groupby('Customer ID')['Order ID'].nunique()

# Name this Series so it becomes a proper column name when merged.
order_freq.name = 'order_frequency'

print("Top 5 customers by order frequency:")
print(order_freq.sort_values(ascending=False).head())


In [ ]:
# --- Average Order Value per Customer ---

# groupby('Customer ID') groups all rows by customer, as before.
#
# ['Sales'] selects the Sales column within each group.
#
# .mean() calculates the arithmetic average of Sales for each customer.
# Formula: sum of all sales / number of rows for that customer.
# This tells us how much a customer typically spends per line item.
avg_order = df.groupby('Customer ID')['Sales'].mean()

# Assign a name so it becomes the column label after merging.
avg_order.name = 'avg_order_value'

print("Top 5 customers by average order value:")
print(avg_order.sort_values(ascending=False).head())


In [ ]:
# --- Combine all three Series into one DataFrame ---

# pd.concat() is a Pandas function that joins multiple data structures together.
#
# [customer_spending, order_freq, avg_order] is a Python list of the three Series.
#
# axis=1 means 'join them side by side as columns'.
#   axis=0 would stack them vertically (add more rows).
#   axis=1 stacks them horizontally (add more columns) — which is what we want.
#
# Because all three Series share the same index (Customer ID),
# Pandas automatically lines them up correctly row by row.
customer_summary = pd.concat([customer_spending, order_freq, avg_order], axis=1)

# .columns assigns new column names to the DataFrame.
# The list order must match the order we passed into pd.concat().
customer_summary.columns = ['total_spending', 'order_frequency', 'avg_order_value']

# .reset_index() moves the Customer ID from being the row label (index)
# into a regular column, making the table easier to read and export.
customer_summary = customer_summary.reset_index()

print(f"Customer summary shape: {customer_summary.shape}")
# .shape returns a tuple (rows, columns) — tells us the size of the DataFrame.


In [ ]:
# --- Inspect the customer summary ---

# display() is a Jupyter/Colab function that renders a DataFrame as a
# nicely formatted HTML table in the output, instead of plain text.
# It only works inside notebooks (not plain Python scripts).

print("First 10 rows of customer summary:")
display(customer_summary.head(10))
# .head(10) returns the first 10 rows. Default is 5 if no number is given.

print("\nSummary statistics:")
# \n inside a string means 'new line' — adds a blank line before the next output.

# .describe() automatically calculates key statistics for every numerical column:
# count  = number of non-null values
# mean   = average
# std    = standard deviation (how spread out the values are)
# min    = smallest value
# 25%    = 25th percentile (lower quarter of data)
# 50%    = median (middle value)
# 75%    = 75th percentile (upper quarter of data)
# max    = largest value
customer_summary.describe()


---
## Task 6: Save Cleaned Data (TODO)

Save the cleaned DataFrame to a new CSV file. Never overwrite the original.

In [ ]:
# --- Save the cleaned files ---

# .to_csv() is a DataFrame method that writes the data to a CSV (Comma-Separated Values) file.
# The string inside the brackets is the filename it will be saved as.
#
# index=False tells Pandas NOT to write the row numbers (0, 1, 2, 3...)
# as an extra column in the CSV. Without this, you'd get an unwanted column called 'Unnamed: 0'.
#
# We save to a NEW filename ('superstore_cleaned.csv') to preserve the original file.
# Never overwrite your raw data — if something goes wrong, you can always start over.
df.to_csv('superstore_cleaned.csv', index=False)
print("Saved: superstore_cleaned.csv — the fully cleaned row-level dataset")

# Save the customer-level summary separately.
customer_summary.to_csv('customer_summary.csv', index=False)
print("Saved: customer_summary.csv — one row per customer with aggregated metrics")


---
## Reflection

Answer these in a text cell or comments:

1. Why did we use median instead of mean for filling numerical missing values?
2. What is the difference between the row-level DataFrame (one row per order) and the customer-level summary? When would you use each?
3. If two rows have identical values in every column, are they always true duplicates? When might they not be?

## Reflection — Answers

**1. Why did we use median instead of mean for filling numerical missing values?**

The **mean** (average) is dragged up or down by extreme outliers. In this dataset the `Profit` column ranges from –6,599 to +8,399 — a handful of huge losses would pull the mean far below what a 'typical' order looks like, making it a misleading fill value. The **median** (the middle value when all numbers are sorted) ignores those extremes completely and stays close to what most orders actually look like, so it is a safer choice.

**2. What is the difference between the row-level DataFrame and the customer-level summary? When would you use each?**

The **row-level DataFrame** (`df`) has one row per order line — every individual product, date, discount, and profit is preserved. Use it when you need transaction-level detail, e.g., which products sell best, or which orders were discounted.

The **customer-level summary** (`customer_summary`) collapses all rows for the same customer into a single row of aggregated metrics. Use it when you care about customer behaviour as a whole — e.g., segmenting customers by spending, or predicting churn.

**3. If two rows have identical values in every column, are they always true duplicates? When might they not be?**

Not always. A customer could legitimately order the exact same product at the same price on the same day — two real purchases that happen to look identical. Duplicates more commonly come from data pipeline bugs (a record being written twice) or copy-paste errors. In this dataset the `Row ID` column is a unique identifier for every row, so any two rows sharing the same Row ID would be a clear system-level duplicate. Always check the business context and unique identifiers before blindly dropping rows.